In [ ]:
# Parameters
seed = 20250417
eventstream_name = "es-appareliq-rt"          # Eventstream with the custom endpoint source below
source_name = "kinetiq-rt-endpoint"
kql_database_name = "kdb_appareliq"           # used only by the validation cell
history_hours = 24                            # backfilled trading hours (baseline for the intraday intent index)
pos_per_hour = 240                            # DS-15 POS lines per hour (store + e-com)
digital_per_hour = 900                        # DS-16 product views / carts / wishlists per hour
inventory_per_hour = 60                       # DS-17 DC and store movements per hour
surge_style = "HM9443"                        # planted demo story: launch style surging in the US (T1)
surge_market = "US"
surge_hours = 2                               # last N hours of the backfill carry the surge
surge_multiplier = 3.0
duplicate_rate = 0.005                        # DQ-01: re-sent events (same event_id)
bad_record_rate = 0.002                       # events without sku/style: kept in rt_events_raw, dropped by the update policies
live_minutes = 0                              # > 0: keep streaming one micro-batch per minute after the backfill
publish = True
validate = True

# 05 - Real-time event simulator (DS-15, DS-16, DS-17) - SYNTHETIC

Publishes synthetic retail events to the `es-appareliq-rt` Eventstream custom endpoint. The Eventstream lands them in
Eventhouse `kdb_appareliq.rt_events_raw`; KQL update policies fan them out to `rt_pos_transaction`, `rt_digital_event`
and `rt_inventory_movement` (OneLake availability on).

* Products, locations, markets and demand weights come from `lh_medallion` Gold, so every event joins to the batch data.
* The last `surge_hours` carry a demand surge for `surge_style` in `surge_market`, which makes the intraday T1 trigger
  (`fn_t1_alerts()`) fire for the demo story.
* Deterministic content (fixed `seed`); timestamps are relative to the run time. Every event has `record_label = SYNTHETIC`.
* The custom-endpoint key is fetched at run time through the Fabric REST API with the caller's identity; no secret is
  stored. Events go out over the Event Hubs REST API with a SAS token built with the standard library (no extra package).

In [ ]:
# Setup: Gold reference data and run context
import base64, datetime as dt, hashlib, hmac, json, time, urllib.parse, urllib.request
import numpy as np
import pandas as pd
import duckdb
import notebookutils

LABEL = "SYNTHETIC"
SCENARIO = "UC1-intraday"
rng = np.random.default_rng(seed)
ws_id = notebookutils.runtime.context["currentWorkspaceId"]
run_start = dt.datetime.utcnow().replace(second=0, microsecond=0)
run_tag = run_start.strftime("%Y%m%d%H%M")

def gold(table, columns=None):
    # Python kernel: DuckDB delta_scan over the default-lakehouse mount (supports deletion vectors; no Spark)
    cols = ", ".join(columns) if columns else "*"
    return duckdb.sql(f"SELECT {cols} FROM delta_scan('/lakehouse/default/Tables/gold/{table}')").df()

product = gold("dim_product")
location = gold("dim_location")
market = gold("dim_market").set_index("market_id")
cand = gold("mart_rebalancing_candidates", ["sku_id", "market_id", "daily_rate_7d"])

selling = location[location.location_type.isin(["STORE", "ECOM"])].reset_index(drop=True)
dcs = location[location.location_type == "DC"].reset_index(drop=True)
stores = location[location.location_type == "STORE"].reset_index(drop=True)
sku_info = product.set_index("sku_id")
sizes_by_style = {s: g.reset_index(drop=True) for s, g in product.groupby("style_id")}
selling_by_market = {m: g.reset_index(drop=True) for m, g in selling.groupby("market_id")}
dc_by_market = {m: g.location_id.tolist() for m, g in dcs.groupby("market_id")}
stores_by_market = {m: g.location_id.tolist() for m, g in stores.groupby("market_id")}
SPAN = 3600   # seconds covered by one emitted batch (3600 for backfill hours, 60 for live minutes)

# Demand weight per SKU-size x market: the Gold 7-day sell rate (+ a floor so every ranged SKU can sell)
w = cand.assign(weight=cand.daily_rate_7d.fillna(0).clip(lower=0) + 0.05).merge(
    product[["sku_id", "style_id"]], on="sku_id")
w = w[w.market_id.isin(market.index)].reset_index(drop=True)
style_w = w.groupby(["style_id", "market_id"], as_index=False).weight.sum()
print(f"{len(product)} SKU-sizes, {len(selling)} selling locations, {len(w)} SKU x market weights; run_tag={run_tag}")


def surge_factor(frame, ts):
    """Multiplier per row: surge_multiplier for the surge style x market during the last surge_hours."""
    hot = ts >= run_start - dt.timedelta(hours=surge_hours)
    return np.where(hot & (frame.style_id == surge_style).values & (frame.market_id == surge_market).values,
                    surge_multiplier, 1.0)


def pick(frame, n, ts):
    p = frame.weight.values * surge_factor(frame, ts)
    return frame.iloc[rng.choice(len(frame), size=n, p=p / p.sum())].reset_index(drop=True)


def stamp(hour_start, n):
    return [hour_start + dt.timedelta(seconds=int(s)) for s in np.sort(rng.integers(0, SPAN, n))]


def iso(ts):
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")

In [ ]:
# DS-15 POS transactions (store + e-com): sale 95 %, return 5 %; local currency price from Gold MSRP x FX x premium
def gen_pos(hour_start, n):
    rows = pick(w, n, hour_start)
    out = []
    for i, (r, ts) in enumerate(zip(rows.itertuples(), stamp(hour_start, n))):
        locs = selling_by_market[r.market_id]
        loc = locs.iloc[int(rng.integers(0, len(locs)))]
        m = market.loc[r.market_id]
        p = sku_info.loc[r.sku_id]
        full = float(p.msrp_usd) * float(m.fx_usd_to_local) * float(m.price_premium)
        price = round(full * (1.0 if p.lifecycle in ("LAUNCH", "CORE") or rng.random() > 0.25 else 0.7), 2)
        etype = "return" if rng.random() < 0.05 else "sale"
        qty = int(rng.choice([1, 1, 1, 2, 3]))
        out.append(dict(event_id=f"POS-{run_tag}-{hour_start:%d%H}-{i:05d}", event_ts=iso(ts), event_category="pos",
                        event_type=etype, market_id=r.market_id, location_id=loc.location_id, to_location_id=None,
                        channel="ECOM" if loc.location_type == "ECOM" else "STORE", sku_id=r.sku_id,
                        style_id=r.style_id, qty=qty, unit_price_local=price, amount_local=round(price * qty, 2),
                        currency=m.currency, session_id=None, reference_id=f"TXN-{run_tag}-{i:05d}",
                        source_system="POS", scenario=SCENARIO, record_label=LABEL))
    return out

In [ ]:
# DS-16 e-commerce intent: product views, add/remove cart, wishlist, notify-me (launch styles only)
DIGITAL_TYPES = ["product_view", "add_to_cart", "remove_from_cart", "wishlist_add", "notify_me"]
DIGITAL_P = [0.72, 0.14, 0.04, 0.07, 0.03]


def gen_digital(hour_start, n):
    rows = pick(style_w, n, hour_start)
    out = []
    for i, (r, ts) in enumerate(zip(rows.itertuples(), stamp(hour_start, n))):
        etype = str(rng.choice(DIGITAL_TYPES, p=DIGITAL_P))
        sizes = sizes_by_style[r.style_id]
        if etype == "notify_me" and not bool(sizes.is_launch.any()):
            etype = "wishlist_add"
        sku = sizes.sku_id.iloc[int(rng.integers(0, len(sizes)))] if etype in ("add_to_cart", "remove_from_cart") else None
        out.append(dict(event_id=f"DIG-{run_tag}-{hour_start:%d%H}-{i:05d}", event_ts=iso(ts), event_category="digital",
                        event_type=etype, market_id=r.market_id, location_id=None, to_location_id=None,
                        channel=str(rng.choice(["APP", "WEB"], p=[0.6, 0.4])), sku_id=sku, style_id=r.style_id, qty=1,
                        unit_price_local=None, amount_local=None, currency=None,
                        session_id=f"S-{run_tag}-{int(rng.integers(0, 10**8)):08d}", reference_id=None,
                        source_system="ECOM", scenario=SCENARIO, record_label=LABEL))
    return out

In [ ]:
# DS-17 DC / store inventory movements: DC receipts, DC->store transfers (out + in pair), wholesale shipments, adjustments
def gen_inventory(hour_start, n):
    rows = pick(w, n, hour_start)
    out = []
    for i, (r, ts) in enumerate(zip(rows.itertuples(), stamp(hour_start, n))):
        base = dict(event_category="inventory", market_id=r.market_id, channel=None, sku_id=r.sku_id,
                    style_id=r.style_id, unit_price_local=None, amount_local=None, currency=None, session_id=None,
                    source_system="WMS", scenario=SCENARIO, record_label=LABEL, to_location_id=None)
        dc, st = dc_by_market.get(r.market_id), stores_by_market.get(r.market_id)
        if not dc or not st:
            continue
        dc_id = dc[int(rng.integers(0, len(dc)))]
        store_id = st[int(rng.integers(0, len(st)))]
        kind = rng.random()
        eid = f"INV-{run_tag}-{hour_start:%d%H}-{i:05d}"
        if kind < 0.25:
            out.append(base | dict(event_id=eid, event_ts=iso(ts), event_type="receipt", location_id=dc_id,
                                   qty=int(rng.choice([12, 24, 36, 48])), reference_id=f"PO-RT-{i:05d}"))
        elif kind < 0.70:
            q = int(rng.integers(2, 13))
            ref = f"TO-{run_tag}-{i:05d}"
            out.append(base | dict(event_id=eid + "-O", event_ts=iso(ts), event_type="transfer_out", location_id=dc_id,
                                   to_location_id=store_id, qty=q, reference_id=ref))
            out.append(base | dict(event_id=eid + "-I", event_ts=iso(ts + dt.timedelta(minutes=int(rng.integers(30, 240)))),
                                   event_type="transfer_in", location_id=store_id, qty=q, reference_id=ref))
        elif kind < 0.85:
            out.append(base | dict(event_id=eid, event_ts=iso(ts), event_type="shipment", location_id=dc_id,
                                   qty=int(rng.integers(6, 49)), reference_id=f"WO-RT-{i:05d}"))
        else:
            out.append(base | dict(event_id=eid, event_ts=iso(ts), event_type="adjustment", location_id=store_id,
                                   qty=int(rng.choice([-3, -2, -1, 1, 2])), reference_id="CYCLE-COUNT"))
    return [e for e in out if e["event_ts"] <= iso(run_start + dt.timedelta(minutes=1))]

In [ ]:
# Data-quality edge cases + Event Hubs publisher (custom endpoint key fetched at run time; nothing stored)
def add_edge_cases(events):
    n = len(events)
    dups = [dict(events[int(k)]) for k in rng.choice(n, size=int(n * duplicate_rate), replace=False)]
    bad = []
    for k in rng.choice(n, size=max(1, int(n * bad_record_rate)), replace=False):
        e = dict(events[int(k)])
        e.update(event_id=e["event_id"] + "-BAD", sku_id=None, style_id=None)
        bad.append(e)
    return events + dups + bad, len(dups), len(bad)


def fabric(path):
    req = urllib.request.Request(f"https://api.fabric.microsoft.com/v1{path}")
    req.add_header("Authorization", "Bearer " + notebookutils.credentials.getToken("pbi"))
    return json.loads(urllib.request.urlopen(req).read().decode())


def endpoint():
    es = next(i for i in fabric(f"/workspaces/{ws_id}/eventstreams")["value"] if i["displayName"] == eventstream_name)
    src = next(s for s in fabric(f"/workspaces/{ws_id}/eventstreams/{es['id']}/topology")["sources"]
               if s["name"] == source_name)
    conn = fabric(f"/workspaces/{ws_id}/eventstreams/{es['id']}/sources/{src['id']}/connection")
    parts = dict(p.split("=", 1) for p in conn["accessKeys"]["primaryConnectionString"].split(";") if "=" in p)
    return dict(fqns=parts["Endpoint"].replace("sb://", "").strip("/"), hub=conn["eventHubName"],
                key_name=parts["SharedAccessKeyName"], key=parts["SharedAccessKey"])


def sas(ep, ttl=3600):
    uri = urllib.parse.quote_plus(f"https://{ep['fqns']}/{ep['hub']}")
    exp = int(time.time()) + ttl
    sig = base64.b64encode(hmac.new(ep["key"].encode(), f"{uri}\n{exp}".encode(), hashlib.sha256).digest())
    return f"SharedAccessSignature sr={uri}&sig={urllib.parse.quote_plus(sig)}&se={exp}&skn={ep['key_name']}"


def send(ep, events, chunk=250):
    url = f"https://{ep['fqns']}/{ep['hub']}/messages?timeout=60&api-version=2014-01"
    token = sas(ep)
    for i in range(0, len(events), chunk):
        body = json.dumps([{"Body": json.dumps(e)} for e in events[i:i + chunk]]).encode()
        req = urllib.request.Request(url, data=body, method="POST")
        req.add_header("Authorization", token)
        req.add_header("Content-Type", "application/vnd.microsoft.servicebus.json")
        for attempt in range(5):
            try:
                urllib.request.urlopen(req, timeout=60)
                break
            except urllib.error.HTTPError as ex:
                if ex.code not in (429, 500, 503) or attempt == 4:
                    raise
                time.sleep(2 ** attempt)
    return len(events)


ep = endpoint() if publish else None
print("custom endpoint resolved:", ep["hub"] if ep else "(publish disabled)")

In [ ]:
# Run: backfill history_hours (the last surge_hours carry the surge), then optionally stream live micro-batches
sent = dict(pos=0, digital=0, inventory=0, duplicates=0, bad=0)
unique_ids = dict(pos=set(), digital=set(), inventory=set())
t0 = time.time()


def emit(hour_start, scale=1.0):
    batch = {"pos": gen_pos(hour_start, max(1, int(pos_per_hour * scale))),
             "digital": gen_digital(hour_start, max(1, int(digital_per_hour * scale))),
             "inventory": gen_inventory(hour_start, max(1, int(inventory_per_hour * scale)))}
    events = []
    for k, ev in batch.items():
        sent[k] += len(ev)
        unique_ids[k].update(e["event_id"] for e in ev)
        events += ev
    events, nd, nb = add_edge_cases(events)
    sent["duplicates"] += nd
    sent["bad"] += nb
    if publish:
        send(ep, events)
    return len(events)


for h in range(history_hours, 0, -1):
    emit(run_start - dt.timedelta(hours=h))
SPAN = 60
for m in range(live_minutes):
    run_start = dt.datetime.utcnow().replace(second=0, microsecond=0)
    emit(run_start - dt.timedelta(minutes=1), scale=1 / 60)
    time.sleep(max(0, 60 - (time.time() - t0) % 60))
print(f"published in {time.time() - t0:.0f}s:", sent)

In [ ]:
# Validate in the Eventhouse: landing count, fan-out, de-duplication, dropped bad records, intraday T1 for the demo story
def kql(query):
    db = next(d for d in fabric(f"/workspaces/{ws_id}/kqlDatabases")["value"] if d["displayName"] == kql_database_name)
    req = urllib.request.Request(db["properties"]["queryServiceUri"] + "/v1/rest/query",
                                 data=json.dumps({"db": kql_database_name, "csl": query}).encode(), method="POST")
    req.add_header("Authorization", "Bearer " + notebookutils.credentials.getToken("kusto"))
    req.add_header("Content-Type", "application/json")
    t = json.loads(urllib.request.urlopen(req).read().decode())["Tables"][0]
    return pd.DataFrame(t["Rows"], columns=[c["ColumnName"] for c in t["Columns"]])


if validate and publish:
    expect = {k: len(v) for k, v in unique_ids.items()}
    q = (f"let r = '{run_tag}';"
         "union (pos_events() | extend t='pos'), (digital_events() | extend t='digital'), (inventory_events() | extend t='inventory')"
         "| where event_id has r | summarize n = count() by t")
    for attempt in range(40):   # Eventstream -> Eventhouse ingestion is batched; allow a few minutes
        got = dict(kql(q).values.tolist())
        if all(got.get(k, 0) >= v for k, v in expect.items()):
            break
        print(f"waiting for ingestion: {got} / {expect}")
        time.sleep(30)
    raw = kql(f"rt_events_raw | where event_id has '{run_tag}' | summarize n = count(), bad = countif(isempty(sku_id) and isempty(style_id))").iloc[0]
    alerts = kql("fn_t1_alerts()")
    hero = alerts[(alerts.style_id == surge_style) & (alerts.market_id == surge_market)]
    intent = kql(f"fn_intent_index() | where style_id == '{surge_style}' and market_id == '{surge_market}'")
    checks = pd.DataFrame([
        ("typed tables hold every unique event (dupes removed)", got == expect, f"{got} vs {expect}"),
        ("raw landing keeps duplicates and bad records", int(raw.n) == sum(expect.values()) + sent["duplicates"] + sent["bad"],
         f"raw={int(raw.n)}"),
        ("bad records stay in raw only", int(raw.bad) == sent["bad"], f"bad={int(raw.bad)}"),
        ("surge raises the intraday intent index >= 1.3", len(intent) == 1 and float(intent.intent_index_intraday.iloc[0]) >= 1.3,
         intent[["intent_index_intraday", "product_views", "carts"]].to_dict("records")),
        ("T1 fires intraday for the surge style x market", len(hero) > 0, f"{len(hero)} SKU-sizes; {len(alerts)} alerts total"),
    ], columns=["check", "passed", "detail"])
    display(checks)
    display(alerts.head(20))
    assert checks.passed.all(), checks[~checks.passed]